# 01_parse_cha_files.ipynb

## Purpose
This notebook parses DementiaBank Pitt Cookie Theft `.cha` files and creates a clean CSV dataset for model training.

It will:
- Read `.cha` files from `Pitt/Dementia/cookie` and `Pitt/Control/cookie`
- Extract participant metadata: age, sex, group, MMSE
- Extract only participant speech lines marked with `*PAR:`
- Remove investigator lines, morphology tiers, grammar tiers, and CHAT symbols
- Match each transcript with its corresponding `.mp3` audio file
- Save `transcripts_dataset.csv`

Expected raw dataset structure:

```text
data/raw/Pitt/
├── Dementia/
│   └── cookie/
│       ├── 001-0.cha
│       ├── 001-0.mp3
│       └── ...
└── Control/
    └── cookie/
        ├── 002-0.cha
        ├── 002-0.mp3
        └── ...
```

## 1. Import required libraries

In [7]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 50)

## 2. Set dataset paths

Change `DATA_ROOT` if your Pitt folder is in another location.

Example for your current structure:

```python
DATA_ROOT = Path(r"C:/Users/YOUR_NAME/Desktop/y4s1/data/Pitt")
```

For project folder structure, use:

```python
DATA_ROOT = Path("data/raw/Pitt")
```

In [10]:
# OPTION 1: Recommended project structure
DATA_ROOT = Path(r"C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt")

# OPTION 2: Use your actual local path if needed
# DATA_ROOT = Path(r"C:/Users/YOUR_NAME/Desktop/y4s1/data/Pitt")

OUTPUT_DIR = Path(r"C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_DIR = DATA_ROOT/ "Control" / "cookie"
DEMENTIA_DIR = DATA_ROOT / "Dementia" / "cookie"

print("DATA_ROOT:", DATA_ROOT.resolve())
print("Control folder exists:", CONTROL_DIR.exists())
print("Dementia folder exists:", DEMENTIA_DIR.exists())

DATA_ROOT: C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt
Control folder exists: True
Dementia folder exists: True


## 3. Check available files

In [11]:
control_cha_files = sorted(CONTROL_DIR.glob("*.cha")) if CONTROL_DIR.exists() else []
dementia_cha_files = sorted(DEMENTIA_DIR.glob("*.cha")) if DEMENTIA_DIR.exists() else []

control_audio_files = sorted(CONTROL_DIR.glob("*.mp3")) if CONTROL_DIR.exists() else []
dementia_audio_files = sorted(DEMENTIA_DIR.glob("*.mp3")) if DEMENTIA_DIR.exists() else []

print("Control CHA files:", len(control_cha_files))
print("Dementia CHA files:", len(dementia_cha_files))
print("Control MP3 files:", len(control_audio_files))
print("Dementia MP3 files:", len(dementia_audio_files))

print("\nSample Control files:")
for f in control_cha_files[:5]:
    print("-", f.name)

print("\nSample Dementia files:")
for f in dementia_cha_files[:5]:
    print("-", f.name)

Control CHA files: 243
Dementia CHA files: 309
Control MP3 files: 243
Dementia MP3 files: 309

Sample Control files:
- 002-0.cha
- 002-1.cha
- 002-2.cha
- 002-3.cha
- 006-2.cha

Sample Dementia files:
- 001-0.cha
- 001-2.cha
- 003-0.cha
- 005-0.cha
- 005-2.cha


## 4. CHAT cleaning functions

CHAT files contain useful speech lines and extra annotation lines.

We keep:
- `*PAR:` participant lines

We remove:
- `*INV:` investigator lines
- `%mor:` morphology tiers
- `%gra:` grammar tiers
- CHAT markers like `[//]`, `[/]`, `[+ gram]`, `&-uh`, `[: out of]`

In [12]:
def clean_chat_text(text: str) -> str:
    """Clean CHAT transcript symbols from one participant utterance."""
    if not isinstance(text, str):
        return ""

    # Replace CHAT replacement annotations like outta [: out of] with the replacement text when possible
    text = re.sub(r"(\S+)\s*\[:\s*([^\]]+)\]", r"\2", text)

    # Remove angle bracket grouping but keep the words inside
    text = text.replace("<", " ").replace(">", " ")

    # Remove common CHAT bracket annotations: [/], [//], [+ gram], [+ exc], etc.
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # Remove fillers/transcriber markers like &-uh, &-um
    text = re.sub(r"&[-\w]+", " ", text)

    # Remove overlap markers and special CHAT symbols
    text = text.replace("+<", " ")
    text = text.replace("xxx", " ")
    text = text.replace("yyy", " ")
    text = text.replace("www", " ")

    # Fix common partial-word markers: tryin(g) -> trying
    text = re.sub(r"\(([^)]*)\)", r"\1", text)

    # Remove punctuation except apostrophes inside words
    text = re.sub(r"[^a-zA-Z0-9'\s]", " ", text)

    # Normalize spaces and lowercase
    text = re.sub(r"\s+", " ", text).strip().lower()

    return text


def extract_participant_text(lines):
    """Extract and clean only *PAR participant utterances."""
    par_lines = []

    for line in lines:
        line = line.strip()
        if line.startswith("*PAR:"):
            utterance = line.replace("*PAR:", "", 1).strip()
            cleaned = clean_chat_text(utterance)
            if cleaned:
                par_lines.append(cleaned)

    return " ".join(par_lines)


def parse_metadata(lines):
    """Extract metadata from @ID and @Media lines."""
    age = None
    sex = None
    group = None
    role = None
    mmse = None
    media_id = None

    for line in lines:
        line = line.strip()

        if line.startswith("@ID:") and "|PAR|" in line:
            # Example:
            # @ID: eng|Pitt|PAR|58;|female|Control||Participant|30||
            parts = line.split("|")
            if len(parts) >= 9:
                age = parts[3].replace(";", "").strip() or None
                sex = parts[4].strip() or None
                group = parts[5].strip() or None
                role = parts[7].strip() or None
                mmse = parts[8].strip() or None

        if line.startswith("@Media:"):
            # Example: @Media: 002-0, audio
            media_part = line.replace("@Media:", "", 1).strip()
            media_id = media_part.split(",")[0].strip()

    return {
        "age": age,
        "sex": sex,
        "group_from_metadata": group,
        "role": role,
        "mmse": mmse,
        "media_id": media_id,
    }


def count_chat_markers(lines):
    """Count useful raw CHAT markers before cleaning for explainable linguistic features."""
    par_text_raw = " ".join(
        line.replace("*PAR:", "", 1).strip()
        for line in lines
        if line.strip().startswith("*PAR:")
    )

    return {
        "num_par_utterances": sum(1 for line in lines if line.strip().startswith("*PAR:")),
        "num_inv_utterances": sum(1 for line in lines if line.strip().startswith("*INV:")),
        "filler_count": len(re.findall(r"&[-\w]+", par_text_raw)),
        "repetition_marker_count": par_text_raw.count("[/]") + par_text_raw.count("[//]"),
        "error_marker_count": len(re.findall(r"\[\+[^\]]+\]", par_text_raw)),
        "replacement_marker_count": len(re.findall(r"\[:[^\]]+\]", par_text_raw)),
    }


def safe_read_cha_file(path: Path):
    """Read CHA file safely using fallback encodings."""
    for encoding in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            return path.read_text(encoding=encoding, errors="ignore").splitlines()
        except Exception:
            continue
    return path.read_text(errors="ignore").splitlines()

## 5. Test parser on one sample file

In [13]:
sample_file = None
if control_cha_files:
    sample_file = control_cha_files[0]
elif dementia_cha_files:
    sample_file = dementia_cha_files[0]

if sample_file is None:
    print("No .cha files found. Please check DATA_ROOT path.")
else:
    lines = safe_read_cha_file(sample_file)
    metadata = parse_metadata(lines)
    transcript = extract_participant_text(lines)
    marker_counts = count_chat_markers(lines)

    print("Sample file:", sample_file.name)
    print("Metadata:", metadata)
    print("Marker counts:", marker_counts)
    print("\nClean transcript preview:\n")
    print(transcript[:1000])

Sample file: 002-0.cha
Metadata: {'age': '58', 'sex': 'female', 'group_from_metadata': 'Control', 'role': 'Participant', 'mmse': '30', 'media_id': '002-0'}
Marker counts: {'num_par_utterances': 18, 'num_inv_utterances': 3, 'filler_count': 5, 'repetition_marker_count': 3, 'error_marker_count': 3, 'replacement_marker_count': 1}

Clean transcript preview:

the scene is in the in the kitchen 3754 5640 the mother is wiping dishes and the water is running on the floor 5776 11843 a child is trying to get a boy is trying to get cookies out of a jar and he's about to tip over on a stool 12138 17928 the little girl is reacting to his falling 19223 23097 it seems to be summer out 25240 26770 the window is open 27416 28166 the curtains are blowing 29559 30632 it must be a gentle breeze 30721 31801 there's grass outside in the garden 32143 33787 mother's finished certain of the the dishes 35151 37890 kitchen's very tidy 39125 40315 the mother seems to have nothing in the house to eat except cookies

## 6. Build complete transcript dataset

In [14]:
def build_transcript_dataset(data_root: Path):
    rows = []

    for label in ["Control", "Dementia"]:
        folder = data_root / label / "cookie"

        if not folder.exists():
            print(f"Warning: folder not found: {folder}")
            continue

        cha_files = sorted(folder.glob("*.cha"))
        print(f"Reading {label}: {len(cha_files)} CHA files")

        for cha_file in cha_files:
            lines = safe_read_cha_file(cha_file)

            metadata = parse_metadata(lines)
            marker_counts = count_chat_markers(lines)
            transcript = extract_participant_text(lines)

            audio_path = cha_file.with_suffix(".mp3")
            audio_exists = audio_path.exists()

            word_count = len(transcript.split()) if transcript else 0
            unique_word_count = len(set(transcript.split())) if transcript else 0
            lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

            row = {
                "file_id": cha_file.stem,
                "label": label,
                "label_id": 0 if label == "Control" else 1,
                "age": metadata.get("age"),
                "sex": metadata.get("sex"),
                "group_from_metadata": metadata.get("group_from_metadata"),
                "role": metadata.get("role"),
                "mmse": metadata.get("mmse"),
                "media_id": metadata.get("media_id"),
                "transcript": transcript,
                "word_count": word_count,
                "unique_word_count": unique_word_count,
                "lexical_diversity": lexical_diversity,
                "cha_path": str(cha_file),
                "audio_path": str(audio_path),
                "audio_exists": audio_exists,
                **marker_counts,
            }

            rows.append(row)

    df = pd.DataFrame(rows)
    return df


df = build_transcript_dataset(DATA_ROOT)

print("\nDataset shape:", df.shape)
display(df.head())

Reading Control: 243 CHA files
Reading Dementia: 309 CHA files

Dataset shape: (552, 22)


,file_id,label,label_id,age,sex,group_from_metadata,role,mmse,media_id,transcript,word_count,unique_word_count,lexical_diversity,cha_path,audio_path,audio_exists,num_par_utterances,num_inv_utterances,filler_count,repetition_marker_count,error_marker_count,replacement_marker_count
0,002-0,Control,0,58,female,Control,Participant,30,002-0,the scene is in the in the kitchen 3754 5640 the mother is wiping dishes and the water is running on the floor 5776 ...,178,119,0.668539,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-0.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-0.mp3,True,18,3,5,3,3,1
1,002-1,Control,0,59,female,Control,Participant,30,002-1,oh i see the sink is running over 3044 5547 i see the stool is tipping over 7371 9595 little boy's trying to get coo...,122,79,0.647541,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-1.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-1.mp3,True,15,5,0,1,2,0
2,002-2,Control,0,60,female,Control,Participant,30,002-2,a boy and a girl are in the kitchen with their mother 11623 14609 and the little boy is getting a cookie for the lit...,181,118,0.651934,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-2.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-2.mp3,True,16,2,7,1,3,0
3,002-3,Control,0,61,female,Control,Participant,28,002-3,okay it was summertime and mother and the children were working in the kitchen 12270 18530 and the window was open a...,190,119,0.626316,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-3.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-3.mp3,True,12,2,4,3,1,0
4,006-2,Control,0,72,male,Control,Participant,NaN,006-2,clears throat wait until i put my glasses on 3540 3930 oh there's a girl reaching for a cookie 5702 8730 a boy is up...,120,86,0.716667,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\006-2.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\006-2.mp3,True,14,2,5,1,2,1


## 7. Basic data validation

In [15]:
if df.empty:
    raise ValueError("Dataset is empty. Check DATA_ROOT and folder structure.")

print("Label distribution:")
display(df["label"].value_counts())

print("\nMissing values:")
display(df.isna().sum())

print("\nAudio file availability:")
display(df["audio_exists"].value_counts())

print("\nWord count summary:")
display(df.groupby("label")["word_count"].describe())

print("\nLexical diversity summary:")
display(df.groupby("label")["lexical_diversity"].describe())

Label distribution:


label
Dementia    309
Control     243
Name: count, dtype: int64


Missing values:


file_id                      0
label                        0
label_id                     0
age                          3
sex                          1
group_from_metadata          3
role                         0
mmse                        92
media_id                     0
transcript                   0
word_count                   0
unique_word_count            0
lexical_diversity            0
cha_path                     0
audio_path                   0
audio_exists                 0
num_par_utterances           0
num_inv_utterances           0
filler_count                 0
repetition_marker_count      0
error_marker_count           0
replacement_marker_count     0
dtype: int64


Audio file availability:


audio_exists
True    552
Name: count, dtype: int64


Word count summary:


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
Control,243.0,132.053498,65.411169,40.0,88.5,118.0,164.0,580.0
Dementia,309.0,122.750809,65.770701,22.0,76.0,108.0,151.0,475.0



Lexical diversity summary:


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
Control,243.0,0.702283,0.077295,0.500000,0.645231,0.700000,0.758872,0.924528
Dementia,309.0,0.702271,0.085467,0.453933,0.642336,0.701031,0.765625,0.903846


## 8. Find possible problematic files

These checks help you identify files with empty transcripts, missing audio, or metadata mismatch.

In [16]:
empty_transcripts = df[df["word_count"] == 0]
missing_audio = df[df["audio_exists"] == False]
metadata_mismatch = df[
    df["group_from_metadata"].notna() &
    (df["group_from_metadata"].str.lower() != df["label"].str.lower())
]

print("Empty transcript files:", len(empty_transcripts))
display(empty_transcripts[["file_id", "label", "cha_path"]].head(20))

print("\nMissing audio files:", len(missing_audio))
display(missing_audio[["file_id", "label", "audio_path"]].head(20))

print("\nMetadata label mismatches:", len(metadata_mismatch))
display(metadata_mismatch[["file_id", "label", "group_from_metadata", "cha_path"]].head(20))

Empty transcript files: 0


,file_id,label,cha_path



Missing audio files: 0


,file_id,label,audio_path



Metadata label mismatches: 306


,file_id,label,group_from_metadata,cha_path
243,001-0,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\001-0.cha
244,001-2,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\001-2.cha
245,003-0,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\003-0.cha
246,005-0,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\005-0.cha
247,005-2,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\005-2.cha
248,007-1,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\007-1.cha
249,007-3,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\007-3.cha
250,010-0,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\010-0.cha
251,010-1,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\010-1.cha
252,010-2,Dementia,ProbableAD,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Dementia\cookie\010-2.cha


## 9. Save processed dataset

In [17]:
# Keep only valid transcript samples for training
df_valid = df[df["word_count"] > 0].copy()

# Convert age and MMSE to numeric where possible
df_valid["age"] = pd.to_numeric(df_valid["age"], errors="coerce")
df_valid["mmse"] = pd.to_numeric(df_valid["mmse"], errors="coerce")

output_path = OUTPUT_DIR / "transcripts_dataset.csv"
df_valid.to_csv(output_path, index=False, encoding="utf-8")

print("Saved:", output_path.resolve())
print("Final valid dataset shape:", df_valid.shape)
display(df_valid.head())

Saved: C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\processed\transcripts_dataset.csv
Final valid dataset shape: (552, 22)


,file_id,label,label_id,age,sex,group_from_metadata,role,mmse,media_id,transcript,word_count,unique_word_count,lexical_diversity,cha_path,audio_path,audio_exists,num_par_utterances,num_inv_utterances,filler_count,repetition_marker_count,error_marker_count,replacement_marker_count
0,002-0,Control,0,58.0,female,Control,Participant,30.0,002-0,the scene is in the in the kitchen 3754 5640 the mother is wiping dishes and the water is running on the floor 5776 ...,178,119,0.668539,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-0.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-0.mp3,True,18,3,5,3,3,1
1,002-1,Control,0,59.0,female,Control,Participant,30.0,002-1,oh i see the sink is running over 3044 5547 i see the stool is tipping over 7371 9595 little boy's trying to get coo...,122,79,0.647541,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-1.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-1.mp3,True,15,5,0,1,2,0
2,002-2,Control,0,60.0,female,Control,Participant,30.0,002-2,a boy and a girl are in the kitchen with their mother 11623 14609 and the little boy is getting a cookie for the lit...,181,118,0.651934,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-2.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-2.mp3,True,16,2,7,1,3,0
3,002-3,Control,0,61.0,female,Control,Participant,28.0,002-3,okay it was summertime and mother and the children were working in the kitchen 12270 18530 and the window was open a...,190,119,0.626316,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-3.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\002-3.mp3,True,12,2,4,3,1,0
4,006-2,Control,0,72.0,male,Control,Participant,NaN,006-2,clears throat wait until i put my glasses on 3540 3930 oh there's a girl reaching for a cookie 5702 8730 a boy is up...,120,86,0.716667,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\006-2.cha,C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\raw\Pitt\Control\cookie\006-2.mp3,True,14,2,5,1,2,1


## 10. Create train/validation/test split

Important: split at file/session level, not sentence level.

For your current setup, each `.cha` file is treated as one sample.

In [21]:
from sklearn.model_selection import train_test_split

SPLIT_DIR = Path(r"C:\Users\krsna\OneDrive\Desktop\aarabi research\data\dementia_speech_research\data\splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_df, temp_df = train_test_split(
    df_valid,
    test_size=0.30,
    random_state=42,
    stratify=df_valid["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label_id"]
)

train_df.to_csv(SPLIT_DIR / "train.csv", index=False, encoding="utf-8")
val_df.to_csv(SPLIT_DIR / "val.csv", index=False, encoding="utf-8")
test_df.to_csv(SPLIT_DIR / "test.csv", index=False, encoding="utf-8")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
display(train_df["label"].value_counts())

print("\nValidation label distribution:")
display(val_df["label"].value_counts())

print("\nTest label distribution:")
display(test_df["label"].value_counts())

Train: (386, 22)
Validation: (83, 22)
Test: (83, 22)

Train label distribution:


label
Dementia    216
Control     170
Name: count, dtype: int64


Validation label distribution:


label
Dementia    47
Control     36
Name: count, dtype: int64


Test label distribution:


label
Dementia    46
Control     37
Name: count, dtype: int64

## 11. Quick preview for modelling

In [22]:
preview_cols = [
    "file_id", "label", "label_id", "age", "sex", "mmse",
    "word_count", "unique_word_count", "lexical_diversity",
    "filler_count", "repetition_marker_count", "audio_exists", "transcript"
]

display(df_valid[preview_cols].sample(min(5, len(df_valid)), random_state=42))

,file_id,label,label_id,age,sex,mmse,word_count,unique_word_count,lexical_diversity,filler_count,repetition_marker_count,audio_exists,transcript
547,704-0,Dementia,1,50.0,male,23.0,73,56,0.767123,1,1,True,well the little kid's falling off his stool 8735 10898 and the mother is having water run over the sink 12951 19691 ...
81,113-0,Control,0,47.0,female,30.0,88,72,0.818182,1,0,True,the little girl's pointing to her mouth 7053 8600 she wants a cookie 8694 9572 the little boy is getting cookies 101...
140,158-1,Control,0,79.0,female,30.0,137,104,0.759124,7,1,True,climbing 3262 3762 dishwashing 3943 4443 pointing 5974 6464 stealing cookies 8407 9367 the wind is blowing outside 1...
79,109-3,Control,0,63.0,female,30.0,135,81,0.600000,10,4,True,mhm there's a boy and a girl and the boy is on the stool taking cookies out of the cookie jar on a stool 4932 13196 ...
272,033-3,Dementia,1,66.0,male,28.0,145,99,0.682759,7,2,True,okay a child falling off a stool in the attempt to reach the cookie jar which it looks like he's knocked the lid off...


## 12. Final notes

Generated files:

```text
data/processed/transcripts_dataset.csv
data/splits/train.csv
data/splits/val.csv
data/splits/test.csv
```

Next notebook:

```text
02_train_transcript_model.ipynb
```

In the next step, train:
- TF-IDF + Logistic Regression
- TF-IDF + SVM
- TF-IDF + Random Forest

Main metrics:
- Accuracy
- Precision
- Recall
- F1-score
- AUC-ROC
- Confusion matrix